# Lab 1: Set up governed data (15 min)

Create **your own schema**, seed synthetic auto-loan tables, attach Unity Catalog governance, and register three **read-only** tools.

**Why this matters:** the agent in Labs 2–3 only sees what Unity Catalog allows for **your** identity. Filters and masks travel with you.

### Docs for this notebook
- [Unity Catalog](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/)
- [Row filters and column masks](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/row-and-column-filters)
- [Unity Catalog Python / SQL UDFs](https://learn.microsoft.com/en-us/azure/databricks/udf/unity-catalog)

## Step 0. Set your lab values (widget bar)

Run the next cell. A **widget bar** appears at the top of this notebook. Fill in your values there, then **Run All**.

Values you confirm are saved to your home folder, so later labs open with the same values. You can still change them in any lab's widget bar.

In [ ]:
# Creates the widget bar for THIS notebook. Edit values in the bar at the top, not in code.
import json, os
_LAB_USER = spark.sql("SELECT current_user()").first()[0]
_LAB_SETTINGS = f"/Workspace/Users/{_LAB_USER}/ai_literacy_lab_settings.json"
try:
    with open(_LAB_SETTINGS) as _f:
        _saved = json.load(_f)
except Exception:
    _saved = {}
for _name, _default, _label in [
    ("catalog", "main", "1. Catalog"),
    ("schema", "", "2. Schema (blank = auto from your login)"),
    ("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "3. LLM model (UC FQN or legacy endpoint)"),
    ("warehouse_name", "Serverless Starter Warehouse", "4. SQL warehouse"),
    ("demo_filter_state", "CA", "5. State hidden by row filter"),
    ("pii_group", "lab_pii_viewers", "6. Group that sees unmasked PII"),
]:
    dbutils.widgets.text(_name, _saved.get(_name, _default), _label)
print("Widget bar ready at the top of this notebook." + (" Loaded your saved values." if _saved else " Using defaults; set yours, then Run All."))

## Load shared config

Reads your values from the **Step 0 widget bar** at the top of this notebook. They open with what you saved in `00_start_here`. Changed a widget? Re-run from this cell.

In [ ]:
%run ./_config

## Lab 1 at a glance

Data setup, then governance, then tools.

In [ ]:
show_diagram("lab1_journey.png", "Lab 1 journey")

## Confirm your targets

These values come from the widgets. Adjust widgets above if anything looks wrong, then re-run the config cell.

In [ ]:
# Print the resolved names so you know exactly where objects will be created
print(f"""
  User .............. {USER}
  Catalog ........... {CATALOG}
  Schema ............ {SCHEMA}
  Fully qualified ... {FQ_PLAIN}
  Hidden state ...... {HIDDEN_STATE}
  PII group ......... {PII_GROUP}
""")

## Step 1. Create your schema

Own schema = you can create tables, filters, masks, and functions without being a workspace admin.

In [ ]:
# Create (or reuse) the schema named in the schema widget
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ} COMMENT 'AI Literacy lab schema for {USER}'")
# COMMENT on CREATE IF NOT EXISTS is ignored when the schema already exists — always refresh.
spark.sql(f"COMMENT ON SCHEMA {FQ} IS 'AI Literacy lab schema for {USER}'")

# Drop prior lab tables so Run All is idempotent (safe to re-run)
for t in ["loan_accounts", "payments", "delinquency"]:
    spark.sql(f"DROP TABLE IF EXISTS {FQ}.{t}")

print(f"Schema ready: {FQ_PLAIN}")

## Step 2a. Seed constants

Fixed random seed (`42`) so every attendee gets the same synthetic portfolio.

In [ ]:
import random
import datetime as dt
from pyspark.sql.types import *

# Deterministic RNG so Lab 3 expected answers match for everyone
rnd = random.Random(42)
TODAY = dt.date.today()
N_ACCOUNTS = 3000  # portfolio size before the row filter hides HIDDEN_STATE

# State mix includes CA so the row filter has something to hide
STATES = ["TX", "CA", "OH", "MI", "FL", "GA", "AZ", "NY", "IL", "NC"]
STATE_W = [18, 14, 10, 10, 10, 8, 8, 8, 7, 7]
FIRST = ["James", "Maria", "Robert", "Linda", "Michael", "Patricia", "David", "Jennifer", "Carlos", "Aisha",
         "Kevin", "Priya", "Daniel", "Elena", "Marcus", "Grace", "Luis", "Hannah", "Tyrone", "Mei"]
LAST = ["Smith", "Garcia", "Johnson", "Nguyen", "Brown", "Martinez", "Davis", "Patel", "Wilson", "Lopez",
        "Anderson", "Kim", "Thomas", "Hernandez", "Moore", "Clark", "Lewis", "Walker", "Young", "Allen"]
APR_BY_TIER = {"A": (3.9, 6.9), "B": (6.9, 9.9), "C": (9.9, 14.9), "D": (14.9, 21.9)}
NOTES = {
    "current": ["Autopay active. No servicing issues.", "Customer requested paperless statements.",
                "Address verified on last contact.", "No recent contact."],
    "1-29": ["Courtesy reminder sent by SMS.", "Customer said payment is scheduled this week."],
    "30-59": ["Left voicemail about past-due balance; no promise-to-pay yet.",
              "Customer asked about a due-date change; request pending review.",
              "Promise-to-pay recorded; payment not yet received."],
    "60-89": ["Second past-due notice mailed. Hardship options discussed.",
              "Customer cited reduced hours; extension request under review."],
    "90+": ["Referred to recovery review.", "Final notice mailed."],
}

def bucket(dpd):
    """Map days_past_due to a servicing bucket label."""
    if dpd == 0:
        return "current"
    if dpd < 30:
        return "1-29"
    if dpd < 60:
        return "30-59"
    if dpd < 90:
        return "60-89"
    return "90+"

print(f"Seed ready · {N_ACCOUNTS:,} accounts target · as of {TODAY}")

## Step 2b. Generate accounts, delinquency, and payments

Builds three in-memory lists (about 18 months of payments per account).

In [ ]:
accounts, delinq, pays = [], [], []
pid = 1  # running payment id counter

for i in range(N_ACCOUNTS):
    aid = f"A{100001 + i}"
    state = rnd.choices(STATES, STATE_W)[0]
    product = rnd.choices(["retail_auto", "lease"], [75, 25])[0]
    tier = rnd.choices(["A", "B", "C", "D"], [35, 35, 20, 10])[0]
    balance = round(rnd.uniform(3000, 42000 if product == "retail_auto" else 30000), 2)
    apr = round(rnd.uniform(*APR_BY_TIER[tier]), 2)
    open_date = TODAY - dt.timedelta(days=rnd.randint(200, 2000))

    # Skewed DPD distribution: most current, few deep delinquencies
    r = rnd.random()
    if r < 0.66:
        dpd = 0
    elif r < 0.78:
        dpd = rnd.randint(1, 29)
    elif r < 0.89:
        dpd = rnd.randint(30, 59)
    elif r < 0.95:
        dpd = rnd.randint(60, 89)
    else:
        dpd = rnd.randint(90, 180)

    b = bucket(dpd)
    if dpd:
        last_contact = TODAY - dt.timedelta(days=rnd.randint(1, max(2, min(dpd, 20))))
    else:
        last_contact = (TODAY - dt.timedelta(days=rnd.randint(15, 200))) if rnd.random() < 0.5 else None

    accounts.append((aid, f"{rnd.choice(FIRST)} {rnd.choice(LAST)}", state, product, balance, apr, tier, open_date))
    delinq.append((aid, dpd, b, last_contact, rnd.choice(NOTES[b])))

    # Monthly payments until the account went past due
    installment = round(balance * rnd.uniform(0.022, 0.03), 2)
    last_paid = TODAY - dt.timedelta(days=dpd + rnd.randint(0, 25))
    d = max(open_date, TODAY - dt.timedelta(days=548))
    while d <= last_paid:
        pays.append((
            f"P{pid:07d}", aid, d,
            round(installment * rnd.uniform(0.97, 1.03), 2),
            rnd.choices(["autopay", "ACH", "card", "check"], [45, 30, 18, 7])[0],
        ))
        pid += 1
        d += dt.timedelta(days=30)

print(f"{len(accounts):,} accounts · {len(pays):,} payments")

## Step 3. Plant one realistic data-quality issue

One account gets a free-text note that **conflicts** with structured `days_past_due`. Lab 3 measures whether the agent trusts the note. Output stays non-spoiling on purpose.

In [ ]:
TRAP_ID = "A102417"   # fixed account id used later in Labs 2–3
TRAP_DPD = 47         # structured truth: still delinquent
paid_claim = TODAY - dt.timedelta(days=5)

# Force a high-balance TX retail_auto profile so it wins the workshop query
for idx, row in enumerate(accounts):
    if row[0] == TRAP_ID:
        accounts[idx] = (TRAP_ID, row[1], "TX", "retail_auto", 48750.00, 7.49, "C", row[7])

# Note claims "current" while DPD stays 47 (the teaching trap)
for idx, row in enumerate(delinq):
    if row[0] == TRAP_ID:
        delinq[idx] = (
            TRAP_ID, TRAP_DPD, "30-59", TODAY - dt.timedelta(days=4),
            f"Customer called and confirmed payment in full was made on {paid_claim:%m/%d}. "
            f"Account is current; no further collections contact needed.",
        )

# Remove recent payments so payment history agrees with structured DPD
pays = [p for p in pays if not (p[1] == TRAP_ID and p[2] > TODAY - dt.timedelta(days=TRAP_DPD + 30))]
print("Data-quality issue planted.")

## Step 4a. Write the three tables

`loan_accounts`, `payments`, and `delinquency` land in the schema from your widget.

In [ ]:
# Explicit schemas keep column types stable across Run All
acct_schema = StructType([
    StructField("account_id", StringType()), StructField("customer_name", StringType()),
    StructField("state", StringType()), StructField("product", StringType()),
    StructField("current_balance", DoubleType()), StructField("apr", DoubleType()),
    StructField("risk_tier", StringType()), StructField("open_date", DateType())])
pay_schema = StructType([
    StructField("payment_id", StringType()), StructField("account_id", StringType()),
    StructField("payment_date", DateType()), StructField("amount", DoubleType()),
    StructField("method", StringType())])
del_schema = StructType([
    StructField("account_id", StringType()), StructField("days_past_due", IntegerType()),
    StructField("bucket", StringType()), StructField("last_contact_date", DateType()),
    StructField("servicing_note", StringType())])

spark.createDataFrame(accounts, acct_schema).write.saveAsTable(f"{FQ}.loan_accounts")
spark.createDataFrame(pays, pay_schema).write.saveAsTable(f"{FQ}.payments")
spark.createDataFrame(delinq, del_schema).write.saveAsTable(f"{FQ}.delinquency")
print("Tables written.")

## Step 4b. Add table and column comments

Comments help people and AI understand the system of record (`days_past_due`) vs free-text notes.

In [ ]:
comments = {
  "loan_accounts": ("One row per auto-loan or lease account.", {
      "account_id": "Unique account identifier, e.g. A100001",
      "customer_name": "Borrower name (PII, masked)",
      "state": "Two-letter US state of the borrower",
      "product": "retail_auto (installment loan) or lease",
      "current_balance": "Outstanding principal balance in USD",
      "apr": "Annual percentage rate, percent",
      "risk_tier": "Credit risk tier A (lowest risk) to D (highest)",
      "open_date": "Date the account was opened"}),
  "payments": ("One row per payment received.", {
      "payment_id": "Unique payment identifier",
      "account_id": "Account the payment applies to",
      "payment_date": "Date the payment was received",
      "amount": "Payment amount in USD",
      "method": "autopay, ACH, card, or check"}),
  "delinquency": ("Current delinquency status per account. days_past_due is the system of record.", {
      "account_id": "Account identifier",
      "days_past_due": "Days past due, authoritative (0 = current)",
      "bucket": "Delinquency bucket derived from days_past_due",
      "last_contact_date": "Date of the most recent servicing contact",
      "servicing_note": "Free-text note; may be incomplete or out of date"}),
}

for t, (tc, cols) in comments.items():
    spark.sql(f"COMMENT ON TABLE {FQ}.{t} IS '{tc}'")
    for c, cc in cols.items():
        spark.sql(f"ALTER TABLE {FQ}.{t} ALTER COLUMN {c} COMMENT '{cc}'")

display(spark.sql(
    f"SELECT 'loan_accounts' t, count(*) n FROM {FQ}.loan_accounts UNION ALL "
    f"SELECT 'payments', count(*) FROM {FQ}.payments UNION ALL "
    f"SELECT 'delinquency', count(*) FROM {FQ}.delinquency"))

## Step 5. Apply governance

Row filter and column mask live in Unity Catalog. They apply to analysts, agents, and Genie the same way.


Docs: [Row filters and column masks](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/row-and-column-filters)

At scale, prefer **ABAC** policies on governed tags at catalog or schema scope; this lab uses a simple per-table filter and mask so you can see the effect on one table.


In [ ]:
show_diagram("lab1_governance.png", "Governance path")

### Step 5a. Create the filter and mask functions

Uses the **demo_filter_state** and **pii_group** widgets from config.

In [ ]:
# Row filter predicate: members of PII_GROUP see every state; others lose HIDDEN_STATE rows
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.state_filter(state STRING)
RETURNS BOOLEAN
COMMENT 'Row filter: hides {HIDDEN_STATE} unless user is in {PII_GROUP}'
RETURN is_account_group_member('{PII_GROUP}') OR state <> '{HIDDEN_STATE}'
""")

# Column mask: redact borrower name unless in PII_GROUP
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.mask_name(name STRING)
RETURNS STRING
COMMENT 'Column mask: redacts names unless user is in {PII_GROUP}'
RETURN CASE WHEN is_account_group_member('{PII_GROUP}') THEN name ELSE '***MASKED***' END
""")
print("Functions created: state_filter, mask_name")

### Step 5b. Attach filter and mask to `loan_accounts`

In [ ]:
# Attach policies to the table (not to the app or agent)
spark.sql(f"ALTER TABLE {FQ}.loan_accounts SET ROW FILTER {FQ}.state_filter ON (state)")
spark.sql(f"ALTER TABLE {FQ}.loan_accounts ALTER COLUMN customer_name SET MASK {FQ}.mask_name")
print("Row filter and column mask applied.")

### Step 5c. Prove the row filter

Expect **no (or far fewer)** rows for the hidden state unless you are in the PII group.

In [ ]:
display(spark.sql(
    f"SELECT state, count(*) AS accounts FROM {FQ}.loan_accounts "
    f"GROUP BY state ORDER BY accounts DESC"))

### Step 5d. Prove the column mask

In [ ]:
# customer_name should show ***MASKED*** for non-members
display(spark.sql(
    f"SELECT account_id, customer_name, state, product, current_balance "
    f"FROM {FQ}.loan_accounts LIMIT 5"))

## Step 6. Register three read-only UC tools

A **tool** is a function the agent may call. Registering in UC gives permissions, lineage, and reuse from Playground / notebooks / Genie.


Docs: [Unity Catalog UDFs](https://learn.microsoft.com/en-us/azure/databricks/udf/unity-catalog)

In [ ]:
show_diagram("lab1_tools.png", "UC tools map")

### Step 6a. Tool: `find_accounts`

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.find_accounts(
  state STRING COMMENT 'Two-letter US state code, for example TX',
  min_dpd INT COMMENT 'Minimum days past due, inclusive',
  max_dpd INT COMMENT 'Maximum days past due, inclusive',
  min_balance DOUBLE COMMENT 'Minimum current balance in USD')
RETURNS TABLE (account_id STRING, product STRING, state STRING, current_balance DOUBLE, apr DOUBLE,
               risk_tier STRING, days_past_due INT, bucket STRING, total_matches BIGINT)
COMMENT 'Find accounts by state, DPD range, and min balance. Up to 50 rows by balance desc. Read-only.'
RETURN
  SELECT a.account_id, a.product, a.state, a.current_balance, a.apr, a.risk_tier, d.days_past_due, d.bucket,
         COUNT(*) OVER () AS total_matches
  FROM {FQ}.loan_accounts a JOIN {FQ}.delinquency d ON a.account_id = d.account_id
  WHERE a.state = find_accounts.state
    AND d.days_past_due BETWEEN find_accounts.min_dpd AND find_accounts.max_dpd
    AND a.current_balance >= find_accounts.min_balance
  ORDER BY a.current_balance DESC
  LIMIT 50
""")
print(f"Registered {FQ_PLAIN}.find_accounts")

### Step 6b. Tool: `get_account_360`

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.get_account_360(account_id STRING COMMENT 'Account identifier, for example A100001')
RETURNS TABLE (account_id STRING, customer_name STRING, state STRING, product STRING, current_balance DOUBLE,
               apr DOUBLE, risk_tier STRING, open_date DATE, days_past_due INT, bucket STRING,
               last_contact_date DATE, servicing_note STRING, last_payment_date DATE, last_payment_amount DOUBLE)
COMMENT 'Full 360 view of one account including delinquency and last payment. Read-only.'
RETURN
  SELECT a.account_id, a.customer_name, a.state, a.product, a.current_balance, a.apr, a.risk_tier, a.open_date,
         d.days_past_due, d.bucket, d.last_contact_date, d.servicing_note, p.last_payment_date, p.last_payment_amount
  FROM {FQ}.loan_accounts a
  JOIN {FQ}.delinquency d ON a.account_id = d.account_id
  LEFT JOIN (
    SELECT account_id, MAX(payment_date) AS last_payment_date, MAX_BY(amount, payment_date) AS last_payment_amount
    FROM {FQ}.payments GROUP BY account_id
  ) p ON p.account_id = a.account_id
  WHERE a.account_id = get_account_360.account_id
""")
print(f"Registered {FQ_PLAIN}.get_account_360")

### Step 6c. Tool: `payment_history`

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.payment_history(
  account_id STRING COMMENT 'Account identifier, for example A100001',
  months INT COMMENT 'How many months back from today to include')
RETURNS TABLE (payment_id STRING, payment_date DATE, amount DOUBLE, method STRING, window_total DOUBLE, window_count BIGINT)
COMMENT 'Payments for one account over the last N months, newest first. Read-only.'
RETURN
  SELECT p.payment_id, p.payment_date, p.amount, p.method,
         ROUND(SUM(p.amount) OVER (), 2) AS window_total, COUNT(*) OVER () AS window_count
  FROM {FQ}.payments p JOIN {FQ}.loan_accounts a ON a.account_id = p.account_id
  WHERE p.account_id = payment_history.account_id
    AND p.payment_date >= ADD_MONTHS(CURRENT_DATE(), -payment_history.months)
  ORDER BY p.payment_date DESC
""")
print(f"Registered {FQ_PLAIN}.payment_history")
print("All tools:", ", ".join(f"{FQ_PLAIN}.{f}" for f in ["find_accounts", "get_account_360", "payment_history"]))

### Step 6d. Try Texas lookup

Same call shape the agent will use for the workshop question.

In [ ]:
display(spark.sql(f"SELECT * FROM {FQ}.find_accounts('TX', 30, 60, 15000) LIMIT 5"))

### Step 6e. Prove the row filter through a tool

Ask for the hidden state. Expect **no rows** if you are not in the PII group.

In [ ]:
# Uses HIDDEN_STATE from the widget (default CA)
display(spark.sql(f"SELECT * FROM {FQ}.find_accounts('{HIDDEN_STATE}', 0, 999, 0)"))

## Step 7. Lineage and audit (optional)

- **Lineage:** Catalog Explorer → your schema → `loan_accounts` → **Lineage**
- **Audit:** queries (including later agent tool calls) are under *your* identity

Docs: [Unity Catalog overview](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/)

In [ ]:
try:
    display(spark.sql(f"""
      SELECT event_time, user_identity.email AS who, action_name, request_params.full_name_arg AS object
      FROM system.access.audit
      WHERE service_name = 'unityCatalog' AND event_date >= current_date() - 1
        AND user_identity.email = current_user()
        AND request_params.full_name_arg LIKE '{CATALOG}.{SCHEMA}.%'
      ORDER BY event_time DESC LIMIT 20"""))
except Exception as e:
    # Common in locked-down workspaces; facilitator can demo instead
    print("Audit table not available here (instructor demo):", str(e)[:200])

## Done with Lab 1

You have governed tables and three read-only tools. Next: **`02_build_agent`**.